In [ ]:
import pandas as pd
from itertools import combinations
from scipy.stats import binomtest
from statsmodels.stats.contingency_tables import cochrans_q
from statsmodels.stats.multitest import multipletests

WORKBOOK = "/Testing_Sample_Detection_Results.xlsx"
TOOLS = ["gptzero", "pangram", "sapling", "isgen"]


def load_data():
    df = pd.read_excel(WORKBOOK, sheet_name="AI_Detection_Results_Merged_300")
    df = df.dropna(subset=["Language", "Label"]).copy()
    df["true_ai"] = df["Label"].ne("AI-Free")
    for tool in TOOLS:
        df[f"{tool}_correct"] = df[f"{tool}_pred_label_50"].eq("AI") == df["true_ai"]
    return df


def cochran_q_by_language(df):
    rows = []
    for lang in ["Arabic", "English"]:
        sub = df[df["Language"] == lang]
        mat = sub[[f"{t}_correct" for t in TOOLS]].astype(int).to_numpy()
        res = cochrans_q(mat)
        rows.append(
            {
                "Language": lang,
                "n_files": len(sub),
                "Q_statistic": float(res.statistic),
                "df": len(TOOLS) - 1,
                "p_value": float(res.pvalue),
            }
        )
    return pd.DataFrame(rows)


def mcnemar_pairwise(df):
    rows = []
    for lang in ["Arabic", "English"]:
        sub = df[df["Language"] == lang]
        pvals = []
        local_rows = []
        for a, b in combinations(TOOLS, 2):
            a_corr = sub[f"{a}_correct"]
            b_corr = sub[f"{b}_correct"]
            a_only = int(((a_corr) & (~b_corr)).sum())
            b_only = int(((~a_corr) & (b_corr)).sum())
            both_correct = int(((a_corr) & (b_corr)).sum())
            both_wrong = int(((~a_corr) & (~b_corr)).sum())
            n_disc = a_only + b_only
            p = (
                1.0
                if n_disc == 0
                else float(
                    binomtest(
                        min(a_only, b_only), n=n_disc, p=0.5, alternative="two-sided"
                    ).pvalue
                )
            )
            local_rows.append(
                {
                    "Language": lang,
                    "Tool_A": a.capitalize() if a != "gptzero" else "GPTZero",
                    "Tool_B": b.capitalize() if b != "gptzero" else "GPTZero",
                    "Accuracy_A_pct": float(a_corr.mean() * 100),
                    "Accuracy_B_pct": float(b_corr.mean() * 100),
                    "A_correct_B_incorrect": a_only,
                    "B_correct_A_incorrect": b_only,
                    "Both_correct": both_correct,
                    "Both_wrong": both_wrong,
                    "Discordant_pairs": n_disc,
                    "p_value_exact": p,
                }
            )
            pvals.append(p)
        adj = multipletests(pvals, method="fdr_bh")[1]
        for row, p_adj in zip(local_rows, adj):
            row["p_value_BH_adjusted"] = float(p_adj)
            if row["Accuracy_A_pct"] > row["Accuracy_B_pct"]:
                row["Higher_accuracy_tool"] = row["Tool_A"]
            elif row["Accuracy_B_pct"] > row["Accuracy_A_pct"]:
                row["Higher_accuracy_tool"] = row["Tool_B"]
            else:
                row["Higher_accuracy_tool"] = "Tie"
            row["Significant_after_BH"] = bool(p_adj < 0.05)
            rows.append(row)
    return pd.DataFrame(rows)


if __name__ == "__main__":
    df = load_data()
    cq = cochran_q_by_language(df)
    mc = mcnemar_pairwise(df)
    cq.to_csv("paper3_cochrans_q_results.csv", index=False)
    mc.to_csv("paper3_mcnemar_pairwise_results.csv", index=False)
    print("Saved paper3_cochrans_q_results.csv")
    print("Saved paper3_mcnemar_pairwise_results.csv")
    print(cq.to_string(index=False))
    print(mc.to_string(index=False))